# Configure

In [13]:
# 1. Go back to Colab's default directory and clean the environment
%cd /content
!rm -rf FFD_system

# 2. Clone only the streamlit branch from GitHub
!git clone -b streamlit https://github.com/AIVIETNAM-AIO-QP09/FFD_system.git
%cd FFD_system

# 3. FIX FOLDER NAME: Create data/raw folder to match 100% with your config.yaml
!mkdir -p data/raw

# 4. Activate Kaggle API Token
import os
os.environ['KAGGLE_API_TOKEN'] = "KGAT_3b7fd74998b8f6c42207756221cc8945"

print("--> Downloading Financial Transactions Dataset from Kaggle...")
# Download and unzip directly into data/raw folder
!kaggle datasets download -d aryan208/financial-transactions-dataset-for-fraud-detection -p data/raw --unzip

# 5. Rename the extracted CSV file to the standard name required by the system
import glob
csv_files = glob.glob("data/raw/*.csv")
if csv_files:
    os.rename(csv_files[0], "data/raw/financial_fraud_detection_dataset.csv")
    print("--> Success! Real dataset is now ready in data/raw/")

# List files to verify
!ls -l data/raw/

/content
Cloning into 'FFD_system'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 25 (delta 4), reused 24 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 9.39 KiB | 1.88 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/FFD_system
--> Downloading Financial Transactions Dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/aryan208/financial-transactions-dataset-for-fraud-detection
License(s): CC0-1.0
100% 277M/277M [00:02<00:00, 138MB/s]

--> Success! Real dataset is now ready in data/raw/
total 777400
-rw-r--r-- 1 root root 796050204 Jun 13 03:28 financial_fraud_detection_dataset.csv


In [14]:
# 1. Always make sure Colab is in the root directory containing run_pipeline.py
%cd /content/FFD_system

# 2. Set PYTHONPATH using Linux command so the system recognizes 'src' package when running external scripts
%env PYTHONPATH=.

# 3. Directly activate run_pipeline.py from outside
!python run_pipeline.py

/content/FFD_system
env: PYTHONPATH=.
2026-06-13 03:28:47,691 - __main__ - INFO - =============================================================
2026-06-13 03:28:47,691 - __main__ - INFO - STARTING FINANCIAL FRAUD DETECTION PIPELINE RUNNER (WEEK 1)
2026-06-13 03:28:47,691 - __main__ - INFO - =============================================================
2026-06-13 03:28:47,693 - src.data_loader - INFO - Configuration successfully loaded from: /content/FFD_system/configs/config.yaml
2026-06-13 03:28:47,693 - src.data_loader - INFO - Initiating data retrieval from: /content/FFD_system/data/raw/financial_fraud_detection_dataset.csv
2026-06-13 03:29:28,124 - src.data_loader - INFO - Data ingested successfully. Shape: (5000000, 18)
2026-06-13 03:29:28,124 - src.preprocessing - INFO - Executing Enterprise Baseline Preprocessing Pipeline...
2026-06-13 03:29:28,124 - src.preprocessing - INFO - Preprocessing: Applying absolute values to clear negative 'time_since_last_transaction' values.
2026-06

In [15]:
!pip install -r requirements.txt

# Execute

In [16]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from src.data_loader import load_config

In [17]:
# 1. Read the configuration file to get the processed data file path
config = load_config("configs/config.yaml")
processed_data_path = config["paths"]["processed_data_path"]

# Convert to absolute path to avoid Colab path errors
full_path = Path("/content/FFD_system") / processed_data_path
print(f"--> Loading clean DataFrame from artifact: {full_path}")
df_baseline = pd.read_csv(full_path)

# 2. Handle data imbalance (Undersampling) to make the model run faster
df_fraud = df_baseline[df_baseline['is_fraud'] == 1]
df_legit = df_baseline[df_baseline['is_fraud'] == 0]

num_fraud = len(df_fraud)
df_legit_sampled = df_legit.sample(n=num_fraud * 1, random_state=42)

# Combine into an optimized reduced dataset
df_subsample = pd.concat([df_fraud, df_legit_sampled]).sample(frac=1, random_state=42)
print(f"--> Total number of samples for quick training: {len(df_subsample)}")

# 3. Extract Features and Target from pipeline
features = [
    'time_since_last_transaction',
    'transaction_hour',
    'is_weekend',
    'amount_log',
    'spending_deviation_score',
    'velocity_score',
    'geo_anomaly_score'
]
X = df_subsample[features]
y = df_subsample['is_fraud'].astype(int)

# 4. Split dataset into Train / Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Train the Baseline classification model (Random Forest)
print("\n--> Training Baseline Model (super fast)...")
baseline_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)

# 6. Predict and generate performance report
y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

print("\n" + "="*55)
print("     BASELINE MODEL EVALUATION RESULTS (COMPLETE)     ")
print("="*55)
print(classification_report(y_test, y_pred, target_names=['Legit (0)', 'Fraud (1)']))
print(f"ROC-AUC Score achieved: {roc_auc_score(y_test, y_proba):.4f}")
print("="*55)


2026-06-13 03:31:32,173 - src.data_loader - INFO - Configuration successfully loaded from: /content/FFD_system/configs/config.yaml
--> Loading clean DataFrame from artifact: /content/FFD_system/data/processed/cleaned_baseline_data.csv
--> Total number of samples for quick training: 359106

--> Training Baseline Model (super fast)...

     BASELINE MODEL EVALUATION RESULTS (COMPLETE)     
              precision    recall  f1-score   support

   Legit (0)       1.00      0.19      0.31     35911
   Fraud (1)       0.55      1.00      0.71     35911

    accuracy                           0.59     71822
   macro avg       0.77      0.59      0.51     71822
weighted avg       0.77      0.59      0.51     71822

ROC-AUC Score achieved: 0.5947
